# Ciencia a tu tesis: recomendación de artículos científicos

Este cuaderno presenta la fuente de datos, la preparación del catálogo, los criterios de similitud y el protocolo de evaluación. La afinidad ordena lecturas, pero no determina la calidad científica.

## 1. Fuente y objetivo

El catálogo combina OpenAlex y SciELO ArticleMeta. OpenAlex aporta amplitud multidisciplinaria y una muestra distribuida entre 26 campos; SciELO suma una muestra por colección con especial presencia latinoamericana e iberoamericana. El proceso normaliza ambos esquemas, conserva la fuente y registra en el manifiesto los filtros y hashes.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from sisreco.settings import CATALOG, DATA_MANIFEST
if not CATALOG.exists(): print('Falta el catálogo: ejecuta python scripts/download_data.py desde la raíz del proyecto.')

## 2. Inspeccionar el catálogo

Se incluyen artículos publicados entre 2016 y 2025 con título y resumen en inglés o español. Se deduplican por fuente, DOI y título. OpenAlex permite excluir registros retractados; SciELO ArticleMeta no informa de forma consistente ese estado, una limitación que debe considerarse al interpretar el catálogo.

In [ ]:
import json
import pandas as pd
if CATALOG.exists():
    catalog = pd.read_parquet(CATALOG)
    print(f'{len(catalog):,} artículos')
    display(catalog.groupby(['field','language']).size().unstack(fill_value=0))
    if DATA_MANIFEST.exists():
        manifest = json.loads(DATA_MANIFEST.read_text(encoding='utf-8'))
        print('Descarga UTC:', manifest['retrieved_at_utc'])
        print('SHA-256:', manifest['catalog_sha256'])

## 3. Representar y comparar textos

E5 recupera los 500 candidatos iniciales por similitud coseno. Un cross-encoder multilingüe compara la descripción con cada candidato y los reordena según su relevancia; en consultas generales se descartan los puntajes negativos. Para las consultas que mencionan agentes de IA y arquitectura empresarial, el sistema comprueba que los artículos cubran ambos conceptos. Si no encuentra esa combinación en el catálogo, etiqueta las coincidencias parciales por concepto. E5 usa los prefijos query: y passage: y admite hasta 512 tokens por texto.

In [ ]:
from sisreco.engine import _load_reranker, _load_sentence_model, load_embeddings, recommend, recommend_pair
if CATALOG.exists():
    model = _load_sentence_model()
    reranker = _load_reranker()
    vectors = load_embeddings(catalog)
    print('Artículos indexados:', vectors.shape[0])
    print('Dimensión del vector:', vectors.shape[1])

## 4. Consulta bilingüe

Este perfil explora las tecnologías educativas y el aprendizaje universitario. Las dos descripciones expresan la misma intención para observar el efecto del idioma en los resultados.

In [ ]:
query_es = 'Mi proyecto estudia cómo las tecnologías digitales educativas se relacionan con los resultados de aprendizaje y la participación de estudiantes universitarios.'
query_en = 'My project studies how digital educational technologies relate to learning outcomes and student engagement in higher education.'
if CATALOG.exists():
    results_es = recommend(catalog, query_es, embeddings=vectors, model=model, reranker=reranker, top_k=5)
    results_en = recommend(catalog, query_en, embeddings=vectors, model=model, reranker=reranker, top_k=5)
    display(results_es[['title','year','field','score','evidence']])
    display(results_en[['title','year','field','score','evidence']])

## 5. Referencia TF-IDF

TF-IDF compara unigramas y bigramas y funciona como referencia para consultas en inglés. Ambos motores comparan los mismos artículos candidatos con la misma descripción. El puntaje de reranking ordena resultados; no es una probabilidad ni una medida de calidad científica.

In [ ]:
if CATALOG.exists():
    semantic_en, tfidf_en = recommend_pair(query_en, catalog, top_k=5, embeddings=vectors, model=model, reranker=reranker)
    print('Semántico'); display(semantic_en[['title','year','field','score']])
    print('TF-IDF'); display(tfidf_en[['title','year','field','score']])

## 6. Evaluación de relevancia

Se preparan seis temas de investigación con consultas en inglés y español. La comparación de motores utiliza consultas en inglés. Califica los primeros cinco resultados con 0 (no pertinente), 1 (parcial) o 2 (directo). El script de evaluación oculta la identidad del motor.

In [ ]:
from sisreco.evaluation import precision_at_k, ndcg_at_k
print('Ejemplo ilustrativo con juicios 2, 1, 0, 0, 0')
print('Precision@5:', precision_at_k([2,1,0,0,0], 5))
print('nDCG@5:', round(ndcg_at_k([2,1,0,0,0], 5), 3))
print('El ejemplo explica las métricas; no representa los resultados del experimento.')

## 7. Arranque en frío

El estudiante puede recibir resultados desde la primera consulta, sin historial de valoraciones. Un artículo nuevo se añade tras validar sus metadatos y regenerar los dos índices.

## 8. Sesgos y limitaciones

- La cobertura de OpenAlex y su clasificación automática pueden favorecer algunas disciplinas.
- El inglés tiene más representación y el rendimiento puede variar entre idiomas.
- Exigir un resumen completo excluye trabajos con metadatos limitados.
- El periodo elegido omite literatura clásica.
- La afinidad temática puede repetir perspectivas y no mide rigor o calidad.

Analiza la composición por disciplina, idioma y año; compara las consultas paralelas y revisa las etiquetas manuales. Se propone sumar fuentes, recuperar trabajos antiguos y explorar diversidad temática.

## 9. Ejecutar

Instala el proyecto con python -m pip install -e '.[dev]'. Prepara los datos con python scripts/download_data.py y los índices con python scripts/build_index.py. La aplicación se inicia con streamlit run app.py. Consulta README.md para instrucciones completas.